# 06 — Resultados

**Qué hace.** Junta los resultados de los notebooks 02–05 en las tablas finales de cada tarea y en las figuras que usan el README y la app.

**Qué lee.** `rutas.resultados_dir` y, para la sección 1, las transcripciones cacheadas en `rutas.cache_dir` (`data/cache/transcripciones/<corrida>/<clip>.json`).

**Qué produce.** Las tablas de resultados de Task 1 y Task 2 para el README y los archivos que carga la app (la app no recalcula nada).

**Corre en local, sin GPU.** Nada de este notebook transcribe: la sección 1 lee el texto que el notebook 02 guardó en la caché y recalcula el WER y el CER con la normalización actual de `src/normalizacion.py`. **Estas son las métricas definitivas de la capa de transcripción**; las de `wer_por_clip.csv` y `mediciones_transcripcion.csv` se calcularon en Colab con la normalización de ese momento. El tiempo y la VRAM sí son de Colab: se midieron al transcribir y se leen de cada JSON.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
from src.config import cargar_config
from src.metricas import intervalo_bootstrap, wer_cer
from src.normalizacion import normalizar_texto
from src.transcripcion import cargar_cache, id_corrida

cfg = cargar_config()
rutas, asr = cfg["rutas"], cfg["asr"]
pd.set_option("display.max_colwidth", None)

## 1. Task 1: capa de transcripción

Modelos, precisiones, CPU/GPU y glosario, con intervalos.

### 1.1 La caché

`cargar_cache` lee todos los JSON de `data/cache/`. El device, la precisión y las versiones de cada corrida son los que reportó la librería en Colab y quedaron guardados en el JSON. La última línea comprueba que en este proceso no se cargó faster-whisper ni CTranslate2: las métricas salen del texto guardado, no de transcribir otra vez.

In [2]:
corridas = cargar_cache(rutas["cache_dir"])
clips = sorted({clip for rs in corridas.values() for clip in rs})
referencias = {clip: (rutas["referencias_dir"] / cfg["archivos"]["patron_referencia"].format(clip=clip)).read_text(encoding="utf-8")
               for clip in clips}

cache = pd.DataFrame([{
    "corrida": corrida, "modelo": r0["modelo"], "precision": r0["compute_type"],
    "device": "/".join(sorted({r["device"] for r in rs.values()})), "glosario": r0["glosario"] or "sin",
    "clips": len(rs), "transcrito": min(r["fecha"] for r in rs.values())[:16].replace("T", " "),
    "faster_whisper": r0["versiones"]["faster_whisper"], "ctranslate2": r0["versiones"]["ctranslate2"],
} for corrida, rs in corridas.items() for r0 in [next(iter(rs.values()))]])
assert not {"faster_whisper", "ctranslate2"} & set(sys.modules), "este notebook no debe transcribir"
print(f"{sum(cache.clips)} transcripciones de {len(cache)} corridas, sin cargar faster-whisper ni CTranslate2")
cache

62 transcripciones de 7 corridas, sin cargar faster-whisper ni CTranslate2


,corrida,modelo,precision,device,glosario,clips,transcrito,faster_whisper,ctranslate2
0,large-v3_float16_cuda_hotwords,large-v3,float16,cuda,hotwords,10,2026-11-01 22:56,1.2.1,4.8.4
1,large-v3_float16_cuda_initial_prompt,large-v3,float16,cuda,initial_prompt,10,2026-11-01 22:53,1.2.1,4.8.4
2,large-v3_float16_cuda_sin-glosario,large-v3,float16,cuda,sin,10,2026-11-01 22:44,1.2.1,4.8.4
3,large-v3_int8_cuda_sin-glosario,large-v3,int8_float16,cuda,sin,10,2026-11-01 22:49,1.2.1,4.8.4
4,small_float16_cuda_sin-glosario,small,float16,cuda,sin,10,2026-11-01 22:45,1.2.1,4.8.4
5,small_int8_cpu_sin-glosario,small,int8_float32,cpu,sin,2,2026-11-01 22:52,1.2.1,4.8.4
6,small_int8_cuda_sin-glosario,small,int8_float16,cuda,sin,10,2026-11-01 22:46,1.2.1,4.8.4


### 1.2 Antes y después del cambio de normalización

- **El cambio.** Whisper escribe "10 mil" donde la referencia dice "diez mil". Antes, text2num leía el "mil" suelto: "10 mil" quedaba "10 1000" y la referencia "10000", así que el WER contaba **dos errores** (una sustitución y una inserción) por una palabra bien oída. Ahora un multiplicador detrás de una cifra se multiplica (`config.yaml → normalizacion.multiplicadores`; casos en el notebook 03, sección 1.1).
- **Antes.** Es la misma normalización con `multiplicadores` vacío, que desactiva solo ese paso. El assert comprueba que con ella se reproducen **exactamente** las 62 filas de `wer_por_clip.csv` calculadas en Colab: la caché es el mismo texto que midió el notebook 02 y la única diferencia entre antes y después es este cambio.
- **Después.** La normalización actual: son las métricas definitivas.

In [3]:
ANTES = {**cfg["normalizacion"], "multiplicadores": {}}

filas = []
for corrida, rs in corridas.items():
    for clip, r in rs.items():
        antes, despues = wer_cer(referencias[clip], r["texto"], ANTES), wer_cer(referencias[clip], r["texto"])
        filas.append({"corrida": corrida, "clip": clip, "modelo": r["modelo"], "precision": r["compute_type"],
                      "device": r["device"], "glosario": r["glosario"] or "sin",
                      "wer_antes": antes["wer"], "wer": despues["wer"], "cer_antes": antes["cer"], "cer": despues["cer"],
                      "duracion_min": round(r["duracion_audio_s"] / 60, 2), "tiempo_s": r["tiempo_s"],
                      "s_por_min_audio": r["segundos_por_minuto"], "vram_pico_mb": r["vram_pico_mb"]})
por_clip = pd.DataFrame(filas)

colab = pd.read_csv(rutas["wer_por_clip"]).merge(por_clip, on=["corrida", "clip"], suffixes=("_colab", ""))
assert len(colab) == len(por_clip) == 62
assert (colab["wer_colab"] == colab["wer_antes"].round(4)).all() and (colab["cer_colab"] == colab["cer_antes"].round(4)).all()
print("OK: con la normalización anterior se reproducen las 62 filas de wer_por_clip.csv calculadas en Colab")

cambian = por_clip[por_clip["wer"].round(4) != por_clip["wer_antes"].round(4)]
print(f"Cambian {len(cambian)} de {len(por_clip)} filas; clips afectados: {sorted(set(cambian['clip']))}")
cambian[["corrida", "clip", "wer_antes", "wer", "cer_antes", "cer"]].round(4)

OK: con la normalización anterior se reproducen las 62 filas de wer_por_clip.csv calculadas en Colab
Cambian 6 de 62 filas; clips afectados: ['e3']


,corrida,clip,wer_antes,wer,cer_antes,cer
2,large-v3_float16_cuda_hotwords,e3,0.0336,0.0268,0.0130,0.0119
12,large-v3_float16_cuda_initial_prompt,e3,0.0403,0.0336,0.0169,0.0158
22,large-v3_float16_cuda_sin-glosario,e3,0.0537,0.0470,0.0265,0.0254
32,large-v3_int8_cuda_sin-glosario,e3,0.0570,0.0503,0.0226,0.0214
42,small_float16_cuda_sin-glosario,e3,0.0839,0.0671,0.0248,0.0203
54,small_int8_cuda_sin-glosario,e3,0.0805,0.0638,0.0277,0.0231


Dónde está la diferencia, en el clip afectado: el mismo fragmento de la referencia y de la transcripción de `large-v3` float16, normalizados antes y después.

In [4]:
clip, corrida = "e3", id_corrida(asr["modelo"], asr["compute_type"]["cuda"], "cuda", None)
for nombre, reglas in [("antes", ANTES), ("después", None)]:
    for fuente, texto in [("referencia", referencias[clip]), ("large-v3", corridas[corrida][clip]["texto"])]:
        normalizado = normalizar_texto(texto, reglas)
        i = normalizado.find("estoy entre")
        print(f"{nombre:>8} · {fuente:<10}: {normalizado[i:i + 45]}")

   antes · referencia: estoy entre 9 y 10000 soles idealmente 9500 b
   antes · large-v3  : estoy entre 9 y 10 1000 soles idealmente 9500
 después · referencia: estoy entre 9 y 10000 soles idealmente 9500 b
 después · large-v3  : estoy entre 9 y 10000 soles idealmente 9500 b


### 1.3 Por corrida

- **WER y CER del corpus**: todas las palabras de los clips de la corrida juntas (jiwer sobre las listas), antes y después.
- **IC 95 %**: bootstrap del WER medio por clip, remuestreando los clips. Con 10 clips el intervalo es ancho.
- **Velocidad y VRAM**: las mediciones de Colab. `vram_corrida_mb` es el pico menos la memoria que ya estaba ocupada antes de cargar el modelo (556 MB del contexto CUDA).

In [5]:
def por_corrida(corrida):
    rs = corridas[corrida]
    cs = list(rs)
    refs, hips = [referencias[c] for c in cs], [rs[c]["texto"] for c in cs]
    antes, despues = wer_cer(refs, hips, ANTES), wer_cer(refs, hips)
    bajo, alto = intervalo_bootstrap(por_clip.loc[por_clip["corrida"] == corrida, "wer"].tolist())
    picos = [r["vram_pico_mb"] for r in rs.values() if r["vram_pico_mb"] is not None]
    bases = [r["vram_base_mb"] for r in rs.values() if r["vram_base_mb"] is not None]
    r0 = rs[cs[0]]
    return {"corrida": corrida, "modelo": r0["modelo"], "precision": r0["compute_type"], "device": r0["device"],
            "glosario": r0["glosario"] or "sin", "clips": len(cs),
            "wer_antes": round(antes["wer"], 4), "wer": round(despues["wer"], 4),
            "ic95_wer_clip": (round(bajo, 4), round(alto, 4)),
            "cer_antes": round(antes["cer"], 4), "cer": round(despues["cer"], 4),
            "s_por_min_audio": round(sum(r["tiempo_s"] for r in rs.values()) / (sum(r["duracion_audio_s"] for r in rs.values()) / 60), 2),
            "vram_pico_mb": max(picos) if picos else None,
            "vram_corrida_mb": round(max(picos) - min(bases), 1) if picos and bases else None}


metricas = pd.DataFrame([por_corrida(c) for c in corridas]).sort_values(["device", "modelo", "glosario", "precision"],
                                                                        ascending=[False, True, False, True])
mediciones_colab = pd.read_csv(rutas["mediciones_transcripcion"]).set_index("corrida")
assert (mediciones_colab.loc[metricas["corrida"], "wer"].values == metricas["wer_antes"].values).all()
metricas.reset_index(drop=True)

,corrida,modelo,precision,device,glosario,clips,wer_antes,wer,ic95_wer_clip,cer_antes,cer,s_por_min_audio,vram_pico_mb,vram_corrida_mb
0,large-v3_float16_cuda_sin-glosario,large-v3,float16,cuda,sin,10,0.0363,0.0356,"(0.0292, 0.0452)",0.0253,0.0252,6.60,5366.2,4810.0
1,large-v3_int8_cuda_sin-glosario,large-v3,int8_float16,cuda,sin,10,0.0414,0.0406,"(0.0332, 0.0531)",0.0280,0.0279,6.38,3862.2,3306.0
2,large-v3_float16_cuda_initial_prompt,large-v3,float16,cuda,initial_prompt,10,0.0398,0.0390,"(0.0296, 0.0506)",0.0263,0.0261,8.12,5462.2,4906.0
3,large-v3_float16_cuda_hotwords,large-v3,float16,cuda,hotwords,10,0.0371,0.0363,"(0.0271, 0.0564)",0.0259,0.0257,7.81,5558.2,5002.0
4,small_float16_cuda_sin-glosario,small,float16,cuda,sin,10,0.0576,0.0557,"(0.0439, 0.069)",0.0216,0.0210,1.91,1334.2,778.0
5,small_int8_cuda_sin-glosario,small,int8_float16,cuda,sin,10,0.0588,0.0568,"(0.0458, 0.0702)",0.0229,0.0223,1.97,1046.2,490.0
6,small_int8_cpu_sin-glosario,small,int8_float32,cpu,sin,2,0.0446,0.0446,"(0.0233, 0.0593)",0.0141,0.0141,31.92,556.2,0.0


### 1.4 Modelo × precisión en GPU, por clip

WER definitivo de cada clip, sin glosario.

In [6]:
sin_glosario = por_clip[(por_clip["device"] == "cuda") & (por_clip["glosario"] == "sin")]
sin_glosario.pivot_table(index="clip", columns=["modelo", "precision"], values="wer").round(3)

modelo    large-v3                small             
precision  float16 int8_float16 float16 int8_float16
clip                                                
e1           0.019        0.022   0.059        0.062
e2           0.042        0.042   0.039        0.030
e3           0.047        0.050   0.067        0.064
e4           0.035        0.035   0.062        0.066
e5           0.032        0.029   0.064        0.068
l1           0.023        0.019   0.023        0.027
l2           0.033        0.061   0.023        0.033
l3           0.067        0.073   0.067        0.067
l4           0.041        0.046   0.071        0.082
l5           0.026        0.053   0.090        0.085

### 1.5 ¿Cambia el texto entre float16 e int8?

Se compara el texto normalizado de las dos precisiones del mismo modelo, clip por clip, y el WER de una contra la otra.

In [7]:
p0, p1 = asr["precisiones_comparacion"][:2]
filas = []
for modelo in asr["modelos_comparacion"]:
    a, b = corridas[id_corrida(modelo, p0, "cuda", None)], corridas[id_corrida(modelo, p1, "cuda", None)]
    for clip in a:
        filas.append({"modelo": modelo, "clip": clip,
                      "texto_identico": normalizar_texto(a[clip]["texto"]) == normalizar_texto(b[clip]["texto"]),
                      "wer_entre_precisiones": round(wer_cer(a[clip]["texto"], b[clip]["texto"])["wer"], 4),
                      f"wer_{p0}": round(wer_cer(referencias[clip], a[clip]["texto"])["wer"], 4),
                      f"wer_{p1}": round(wer_cer(referencias[clip], b[clip]["texto"])["wer"], 4)})
precisiones = pd.DataFrame(filas)
precisiones.groupby("modelo").agg(clips_identicos=("texto_identico", "sum"), clips=("clip", "count"),
                                  wer_entre_precisiones_medio=("wer_entre_precisiones", "mean")).round(4)

,clips_identicos,clips,wer_entre_precisiones_medio
modelo,,,
large-v3,1,10,0.0155
small,0,10,0.0175


### 1.6 CPU contra GPU

El mismo modelo y la misma precisión (`asr.cpu`) en los dos devices, en los clips de `asr.cpu.clips`. El device es el que reportó la librería al transcribir.

In [8]:
cpu = asr["cpu"]
en_cpu = corridas[id_corrida(cpu["modelo"], cpu["compute_type"], "cpu", None)]
en_gpu = corridas[id_corrida(cpu["modelo"], cpu["compute_type"], "cuda", None)]
cpu_gpu = pd.DataFrame([{
    "clip": clip, "device_cpu": en_cpu[clip]["device"], "device_gpu": en_gpu[clip]["device"],
    "s_por_min_cpu": en_cpu[clip]["segundos_por_minuto"], "s_por_min_gpu": en_gpu[clip]["segundos_por_minuto"],
    "veces_mas_rapido_gpu": round(en_cpu[clip]["tiempo_s"] / en_gpu[clip]["tiempo_s"], 1),
    "texto_identico": normalizar_texto(en_cpu[clip]["texto"]) == normalizar_texto(en_gpu[clip]["texto"]),
    "wer_cpu": round(wer_cer(referencias[clip], en_cpu[clip]["texto"])["wer"], 4),
    "wer_gpu": round(wer_cer(referencias[clip], en_gpu[clip]["texto"])["wer"], 4),
} for clip in cpu["clips"]])
assert (cpu_gpu["device_cpu"] == "cpu").all() and (cpu_gpu["device_gpu"] == "cuda").all()
cpu_gpu

,clip,device_cpu,device_gpu,s_por_min_cpu,s_por_min_gpu,veces_mas_rapido_gpu,texto_identico,wer_cpu,wer_gpu
0,e1,cpu,cuda,35.103,2.432,14.4,False,0.0593,0.0620
1,l1,cpu,cuda,28.458,1.940,14.7,False,0.0233,0.0272


### 1.7 Glosario: initial_prompt contra hotwords

- **Diferencia pareada**: WER del clip con el mecanismo menos WER del mismo clip sin glosario (`large-v3` float16). Negativa es mejora.
- **IC 95 %**: bootstrap de la diferencia media, remuestreando los clips; antes y después del cambio de normalización.
- **Términos**: de los términos del glosario que aparecen en las referencias, cuántos aparecen también en la transcripción (los dos textos normalizados).
- **Diferencia con Colab en el cuarto decimal del IC "antes".** Es el mismo cálculo con la misma semilla, pero el bootstrap depende del orden de los valores: en Colab los clips iban en el orden de la corrida (primero e1 y l1, de la prueba rápida) y aquí van ordenados. La diferencia media sí es idéntica.

In [9]:
GLOSARIO = [*asr["glosario"]["tecnologias"], *asr["glosario"]["restaurante"]]
precision = asr["compute_type"]["cuda"]
base = id_corrida(asr["modelo"], precision, "cuda", None)


def terminos(texto):
    t = f" {normalizar_texto(texto)} "
    return {g for g in GLOSARIO if f" {normalizar_texto(g)} " in t}


wer_de = por_clip.set_index(["corrida", "clip"])
filas = []
for mecanismo in ["sin", *asr["glosario_mecanismos_comparacion"]]:
    corrida = base if mecanismo == "sin" else id_corrida(asr["modelo"], precision, "cuda", mecanismo)
    rs = corridas[corrida]
    fila = {"mecanismo": mecanismo, "wer_corpus": metricas.set_index("corrida").loc[corrida, "wer"]}
    if mecanismo != "sin":
        for cuando, columna in [("antes", "wer_antes"), ("después", "wer")]:
            dif = [wer_de.loc[(corrida, c), columna] - wer_de.loc[(base, c), columna] for c in rs]
            bajo, alto = intervalo_bootstrap(dif)
            fila[f"dif_media_{cuando}"] = round(sum(dif) / len(dif), 4)
            fila[f"ic95_{cuando}"] = (round(bajo, 4), round(alto, 4))
        fila["clips_mejoran"], fila["clips_empeoran"] = sum(d < 0 for d in dif), sum(d > 0 for d in dif)
    dichos = {c: terminos(referencias[c]) for c in rs}
    fila["terminos_bien"] = f"{sum(len(dichos[c] & terminos(r['texto'])) for c, r in rs.items())} de {sum(map(len, dichos.values()))}"
    fila["terminos_perdidos"] = sorted({t for c, r in rs.items() for t in dichos[c] - terminos(r["texto"])})
    filas.append(fila)
glosario = pd.DataFrame(filas)
glosario

,mecanismo,wer_corpus,terminos_bien,terminos_perdidos,dif_media_antes,ic95_antes,dif_media_después,ic95_después,clips_mejoran,clips_empeoran
0,sin,0.0356,39 de 40,[Nagios],NaN,NaN,NaN,NaN,NaN,NaN
1,initial_prompt,0.0390,39 de 40,[Plin],0.0031,"(-0.0093, 0.017)",0.0031,"(-0.0093, 0.017)",4.0,4.0
2,hotwords,0.0363,40 de 40,[],0.0042,"(-0.0083, 0.0171)",0.0042,"(-0.0083, 0.0171)",4.0,6.0


### 1.8 Clips con el WER más alto, y exportar

Primero en la configuración del motor (`asr.modelo`, la precisión de `cuda` y `asr.glosario_mecanismo`), después en todas las corridas. Al final se escriben las tablas definitivas: `metricas_transcripcion.csv` (una fila por corrida) y `metricas_por_clip.csv` (una por corrida y clip).

In [10]:
motor = id_corrida(asr["modelo"], asr["compute_type"]["cuda"], "cuda", asr["glosario_mecanismo"])
print("Configuración del motor:", motor)
display(por_clip[por_clip["corrida"] == motor].sort_values("wer", ascending=False)[["clip", "wer", "cer"]].round(4).head(5))
por_clip.sort_values("wer", ascending=False)[["corrida", "clip", "wer", "cer"]].round(4).head(8)

Configuración del motor: large-v3_float16_cuda_hotwords


,clip,wer,cer
8,l4,0.0867,0.0726
7,l3,0.0725,0.0656
3,e4,0.0586,0.0415
6,l2,0.0469,0.0327
9,l5,0.0370,0.0277


,corrida,clip,wer,cer
49,small_float16_cuda_sin-glosario,l5,0.0899,0.0382
8,large-v3_float16_cuda_hotwords,l4,0.0867,0.0726
61,small_int8_cuda_sin-glosario,l5,0.0847,0.0372
60,small_int8_cuda_sin-glosario,l4,0.0816,0.0452
7,large-v3_float16_cuda_hotwords,l3,0.0725,0.0656
37,large-v3_int8_cuda_sin-glosario,l3,0.0725,0.0598
18,large-v3_float16_cuda_initial_prompt,l4,0.0714,0.0471
48,small_float16_cuda_sin-glosario,l4,0.0714,0.0415


In [11]:
metricas.to_csv(rutas["metricas_transcripcion"], index=False)
por_clip.round(4).to_csv(rutas["metricas_por_clip"], index=False)
print("Escrito:", rutas["metricas_transcripcion"].relative_to(RAIZ), "y", rutas["metricas_por_clip"].relative_to(RAIZ))

Escrito: data/resultados/metricas_transcripcion.csv y data/resultados/metricas_por_clip.csv


## 2. Task 1: capa de registro

Por campo y por método de extracción.

*Pendiente.*

## 3. Task 2: curva de degradación

WER y exactitud por nivel.

*Pendiente.*

## 4. Task 2: validación

Atrapados contra no atrapados y exactitud del diagnóstico.

*Pendiente.*

## 5. Lo que la muestra no permite afirmar

Tamaño de muestra, intervalos y limitaciones.

*Pendiente.*

## 6. Exportar para README y app

Escribir las tablas finales en `rutas.resultados_dir`. La sección 1 ya escribe las suyas (1.8): `metricas_transcripcion.csv` y `metricas_por_clip.csv`.

*Pendiente el resto.*